# Frame Review Tool

**Purpose**: Scroll through frames from a run, filter by metrics or strata, and view predictions against ground truth. Allows marking frames locally as problematic for offline review without modifying the original dataset.

**Inputs**: Local `JPEGImages/` and `SegmentationClass/` caches, plus `per_image_results.csv` from a run.

**Outputs**: A local `problematic_frames.csv` listing flagged frames.

## 1. Install dependencies
Run this cell to install the packages required for this notebook.

In [ ]:
%pip install -q matplotlib pandas opencv-python-headless ipywidgets

import cv2, matplotlib, pandas as pd, numpy as _np, ipywidgets
print(f"cv2 {cv2.__version__} | matplotlib {matplotlib.__version__} "
      f"| pandas {pd.__version__} | numpy {_np.__version__} | ipywidgets {ipywidgets.__version__}")


## 2. Setup and configuration
Resolves local images and the run's results CSV. Override with environment variables if the files move.

In [ ]:
import os
import json
import shutil
import subprocess
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import ipywidgets as W
from IPython.display import display

# ── Paths ────────
# Override with environment variables if the files move.
CKPT_BUCKET = os.getenv("LAP_CKPT_BUCKET", "training-checkpoints-REPLACE_K_NUMBER")
CACHE_DIR = Path(os.getenv("LAP_CACHE_DIR", Path.home() / ".cache" / "laparoscopic_seg"))
DEFAULT_DATA_ROOT = Path.home() / "Desktop" / "Individual Project" / "combined"
IMAGES_DIR = Path(os.getenv("LAP_IMAGES_DIR", DEFAULT_DATA_ROOT / "JPEGImages"))
MASKS_DIR = Path(os.getenv("LAP_MASKS_DIR", DEFAULT_DATA_ROOT / "SegmentationClass"))
OUTPUT_CSV = CACHE_DIR / "problematic_frames.csv"

RUN_URI = os.getenv(
    "LAP_REVIEW_RUN",
    f"gs://{CKPT_BUCKET}/experiments/"
    "baseline-seed2026_baseline_Unet_resnet34_bce_lr1e-04_p20/20260812-205137/per_image_results.csv"
)

def fetch_csv(uri):
    if uri.startswith("gs://"):
        local = CACHE_DIR / uri.split("/")[-2] / "per_image_results.csv"
        if not local.exists():
            local.parent.mkdir(parents=True, exist_ok=True)
            subprocess.run(["gsutil", "-q", "cp", uri, str(local)], check=True)
        return local
    return Path(uri)

csv_path = fetch_csv(RUN_URI)
df = pd.read_csv(csv_path)
print(f"Loaded {len(df)} frames from {RUN_URI}")


## 3. Interactive Review
Filters frames based on IoU and displays them. Click **Mark Problematic** to flag them locally.

In [ ]:
df_sorted = df.sort_values("iou", ascending=True).reset_index(drop=True)
flagged_frames = set()
if OUTPUT_CSV.exists():
    flagged_frames = set(pd.read_csv(OUTPUT_CSV)["key"])

frame_idx = W.IntSlider(min=0, max=len(df_sorted)-1, step=1, value=0, description="Rank:")
out = W.Output()
status_lbl = W.Label(value=f"Flagged {len(flagged_frames)} frames locally.")
btn_flag = W.Button(description="Mark Problematic", button_style="warning")

def render_frame(idx):
    out.clear_output(wait=True)
    row = df_sorted.iloc[idx]
    key = row["key"]
    
    img_path = IMAGES_DIR / f"{key}.jpg"
    mask_path = MASKS_DIR / f"{key}.png"
    
    with out:
        if not img_path.exists():
            print(f"Image not found locally: {img_path}\nSet LAP_IMAGES_DIR or sync it.")
            return
            
        img = cv2.imread(str(img_path))[..., ::-1]
        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE) if mask_path.exists() else np.zeros(img.shape[:2])
        
        fig, axes = plt.subplots(1, 2, figsize=(10, 5))
        axes[0].imshow(img)
        axes[0].set_title(f"{key}\nIoU: {row['iou']:.4f}")
        axes[0].axis("off")
        
        axes[1].imshow(img)
        axes[1].imshow(mask, alpha=0.4, cmap="Reds")
        axes[1].set_title(f"Ground Truth Mask\nSize Class: {row['size_class']}")
        axes[1].axis("off")
        
        plt.tight_layout()
        plt.show()
        
        flag_txt = "[ALREADY FLAGGED]" if key in flagged_frames else ""
        print(f"Thin Class: {row['thin_class']} | No-tool FP: {row['notool_fp']} {flag_txt}")

def on_flag(_):
    key = df_sorted.iloc[frame_idx.value]["key"]
    if key not in flagged_frames:
        flagged_frames.add(key)
        pd.DataFrame({"key": list(flagged_frames)}).to_csv(OUTPUT_CSV, index=False)
        status_lbl.value = f"Flagged {len(flagged_frames)} frames locally."
        render_frame(frame_idx.value)

frame_idx.observe(lambda change: render_frame(change.new), names="value")
btn_flag.on_click(on_flag)

display(W.VBox([
    W.HBox([frame_idx, btn_flag, status_lbl]),
    out
]))
render_frame(0)
